<a href="https://colab.research.google.com/github/be-ayush/ai-ml-learning/blob/main/HOML/HOML3_Chapter6_DimentionalityReduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## Finding Min Number of Dimensions for MNIST

from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
import numpy as np

mnist = fetch_openml('mnist_784', as_frame=False)
X_train, y_train = mnist.data[:60000], mnist.target[:60000]
X_test, y_test = mnist.data[60000:], mnist.target[:60000]

pca = PCA()
pca.fit(X_train)
cumulative_sum = np.cumsum(pca.explained_variance_ratio_)
d = np.argmax(cumulative_sum >= 0.95) + 1

In [ ]:
print(d)

153


In [ ]:
pca = PCA(n_components = 0.95)
X_reduced = pca.fit_transform(X_train)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import make_pipeline

classifier = make_pipeline(PCA(n_components = 0.95), RandomForestClassifier(random_state = 42))

param_distribution = {
    'pca__n_components': np.arange(10,80),
    'randomforestclassifier__n_estimators': np.arange(50, 500)
}

random_search = RandomizedSearchCV(classifier, param_distribution, n_iter = 10, cv = 3, verbose = 1, n_jobs = -1, random_state = 42)
random_search.fit(X_train[:1000], y_train[:1000])

Fitting 3 folds for each of 10 candidates, totalling 30 fits


RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('pca', PCA(n_components=0.95)),
                                             ('randomforestclassifier',
                                              RandomForestClassifier(random_state=42))]),
                   n_jobs=-1,
                   param_distributions={'pca__n_components': array([10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26,
       27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43,
       44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 5...
       414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426,
       427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439,
       440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452,
       453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465,
       466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478,
       479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491,
       492, 493, 494, 495, 496, 497, 498, 499])},
                   random_state=42, verbose=1)

In [ ]:
print(random_search.best_params_)

{'randomforestclassifier__n_estimators': np.int64(314), 'pca__n_components': np.int64(36)}


In [1]:
from sklearn.datasets import make_swiss_roll
from sklearn.manifold import LocallyLinearEmbedding

X_swiss, t = make_swiss_roll(n_samples = 1000, noise = 0.2, random_state = 42)
lle = LocallyLinearEmbedding(n_neighbors = 10, n_components = 2, random_state = 42)

X_unrolled = lle.fit_transform(X_swiss)

In [4]:
from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier

mnist = fetch_openml('mnist_784', as_frame=False)
X_train, y_train = mnist.data[:60000], mnist.target[:60000]
X_test, y_test = mnist.data[60000:], mnist.target[60000:]
random_forest_without_pca = RandomForestClassifier(random_state = 42)

import time
start_time = time.time()
random_forest_without_pca.fit(X_train, y_train)
end_time = time.time()
print(f"Execution time: {end_time - start_time} seconds")

random_forest_without_pca_score = random_forest_without_pca.score(X_test, y_test)
print(random_forest_without_pca_score)

Execution time: 43.86570072174072 seconds
0.9705


In [5]:
pca = PCA(n_components = 0.95)
X_reduced = pca.fit_transform(X_train)
random_forest_with_pca = RandomForestClassifier(random_state = 42)
start_time = time.time()
random_forest_with_pca.fit(X_reduced, y_train)
end_time = time.time()
print(f"Execution time: {end_time - start_time} seconds")

random_forest_with_pca_score = random_forest_with_pca.score(pca.transform(X_test), y_test)
print(random_forest_with_pca_score)

Execution time: 153.0878918170929 seconds
0.9488


In [6]:
sgd_classifier = SGDClassifier(random_state = 42)
start_time = time.time()
sgd_classifier.fit(X_train, y_train)
end_time = time.time()
print(f"Execution time: {end_time - start_time} seconds")

sgd_classifier_score = sgd_classifier.score(X_test, y_test)
print(sgd_classifier_score)

start_time = time.time()
sgd_classifier.fit(X_reduced, y_train)
end_time = time.time()
print(f"Execution time: {end_time - start_time} seconds")

sgd_classifier_score = sgd_classifier.score(pca.transform(X_test), y_test)
print(sgd_classifier_score)

Execution time: 132.16622686386108 seconds
0.874
Execution time: 38.74520993232727 seconds
0.8959


In [7]:
from sklearn.manifold import TSNE
import plotly.express as px
import pandas as pd

# Define the number of images to use (t-SNE can be slow on the full dataset)
n_samples = 5000

# Select the first n images and labels
X_subset = X_train[:n_samples]
y_subset = y_train[:n_samples]

# Initialize and fit t-SNE
# We use n_components=2 to reduce it to 2 dimensions
tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X_subset)

# Create a DataFrame for plotting
tsne_df = pd.DataFrame(data=X_tsne, columns=['TSNE1', 'TSNE2'])
tsne_df['Digit'] = y_subset

# Plot using Plotly
fig = px.scatter(tsne_df, x='TSNE1', y='TSNE2', color='Digit',
                 title=f't-SNE visualization of the first {n_samples} MNIST images',
                 opacity=0.7)
fig.show()

In [8]:
from sklearn.decomposition import PCA
from sklearn.manifold import LocallyLinearEmbedding, MDS

# Use a smaller subset for comparison because MDS is computationally expensive
n_comparison = 5000
X_comp = X_subset[:n_comparison]
y_comp = y_subset[:n_comparison]

# --- 1. PCA ---
print("Running PCA...")
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_comp)

fig_pca = px.scatter(x=X_pca[:, 0], y=X_pca[:, 1], color=y_comp,
                     title=f'PCA (n={n_comparison})', opacity=0.7,
                     labels={'x': 'PC1', 'y': 'PC2'})
fig_pca.show()

# --- 2. LLE ---
print("Running LLE...")
lle = LocallyLinearEmbedding(n_components=2, n_neighbors=10, random_state=42)
X_lle = lle.fit_transform(X_comp)

fig_lle = px.scatter(x=X_lle[:, 0], y=X_lle[:, 1], color=y_comp,
                     title=f'LLE (n={n_comparison})', opacity=0.7,
                     labels={'x': 'Component 1', 'y': 'Component 2'})
fig_lle.show()

# --- 3. MDS ---
print("Running MDS...")
mds = MDS(n_components=2, random_state=42, normalized_stress='auto')
X_mds = mds.fit_transform(X_comp)

fig_mds = px.scatter(x=X_mds[:, 0], y=X_mds[:, 1], color=y_comp,
                     title=f'MDS (n={n_comparison})', opacity=0.7,
                     labels={'x': 'Dimension 1', 'y': 'Dimension 2'})
fig_mds.show()

Running PCA...


Running LLE...


Running MDS...
